# Simple Chat Agent - Workshop Walkthrough

This notebook walks through building a simple chat agent and progressively adding:
1. **State A**: Basic chat agent with an LLM in a loop
2. **State B**: Simulation with FutureAGI Simulate SDK
3. **Evaluation**: Using FutureAGI's built-in eval templates
4. **Optimization**: Improving the prompt with MetaPromptOptimizer

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

## State A: Basic Chat Agent

A simple LLM chat loop. Notice the **deliberately vague** system prompt.

In [ ]:
from config import get_llm_client, get_model_name

# This prompt is intentionally suboptimal
SYSTEM_PROMPT = "You are an AI assistant. Answer user questions. Provide information when asked."

client = get_llm_client()
model = get_model_name()
print(f"Using model: {model}")

In [ ]:
# Test the basic agent with a sample question
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Quick - pros and cons of microservices vs monolith? Meeting in 10 min."}
    ],
)

print("Response:")
print(response.choices[0].message.content)
print("\n--- Notice: Is this concise enough for someone in a rush? ---")

## State B: Add FutureAGI Simulation

We wrap the agent with `AgentWrapper` and run simulated conversations with diverse personas.

In [ ]:
from fi.simulate import (
    AgentWrapper, AgentInput, AgentResponse,
    TestRunner, Scenario, Persona,
)

class ChatAgentWrapper(AgentWrapper):
    """Wraps our chat agent for the Simulate SDK."""
    def __init__(self):
        self.client = get_llm_client()
        self.model = get_model_name()

    async def call(self, input: AgentInput) -> AgentResponse:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + input.messages
        response = self.client.chat.completions.create(model=self.model, messages=messages)
        return AgentResponse(content=response.choices[0].message.content)

In [ ]:
# Define test personas that will expose prompt weaknesses
scenario = Scenario(
    name="simple-chat-test",
    description="Test with diverse user personas",
    dataset=[
        Persona(
            persona={"name": "Priya", "age": 20, "role": "student", "mood": "curious"},
            situation="Needs a concise explanation of quantum entanglement for exam prep.",
            outcome="Clear, concise explanation without jargon.",
        ),
        Persona(
            persona={"name": "Marcus", "age": 35, "role": "engineer", "mood": "frustrated"},
            situation="Laptop keeps crashing during meetings, needs urgent help.",
            outcome="Empathetic, step-by-step troubleshooting.",
        ),
        Persona(
            persona={"name": "Alex", "age": 28, "role": "CEO", "mood": "rushed"},
            situation="Needs quick microservices vs monolith summary for board meeting in 10 min.",
            outcome="Rapid bullet-point response without fluff.",
        ),
    ],
)

In [ ]:
# Run simulation
wrapper = ChatAgentWrapper()
runner = TestRunner()

report = await runner.run_test(
    run_test_name="simple-chat-notebook",
    agent_callback=wrapper,
    scenario=scenario,
)

for result in report.results:
    print(f"\n--- {result.persona.persona['name']} ---")
    print(result.transcript[:500])
    print()

## Evaluation

Now let's evaluate the transcripts using FutureAGI's built-in templates.

In [ ]:
from fi.simulate import evaluate_report

report = evaluate_report(
    report,
    eval_templates=["task_completion", "is_helpful", "is_concise", "tone"],
    model_name="turing_flash",
)

# Display results
for result in report.results:
    name = result.persona.persona["name"]
    print(f"\n--- {name} ---")
    if result.evaluation:
        for template, scores in result.evaluation.items():
            score = scores.get("score", "N/A")
            reason = scores.get("reason", "")[:150]
            print(f"  {template}: {score} - {reason}")

## Optimization

Let's use **MetaPromptOptimizer** to improve the system prompt.
It will analyze failures and rewrite the prompt iteratively.

In [ ]:
from config import get_litellm_model
from fi.opt.generators import LiteLLMGenerator
from fi.opt.optimizers import MetaPromptOptimizer
from fi.opt.base.evaluator import Evaluator
from fi.opt.datamappers import BasicDataMapper
from fi.evals.metrics import CustomLLMJudge
from fi.evals.llm import LiteLLMProvider

litellm_model = get_litellm_model()

# Dataset for optimization
dataset = [
    {"question": "What is quantum entanglement?", "answer": "Two particles linked so measuring one affects the other instantly, regardless of distance."},
    {"question": "My laptop keeps crashing. Help!", "answer": "Let's fix this: 1) Check RAM usage, 2) Update apps, 3) Check temperature. Which first?"},
    {"question": "Quick pros/cons of microservices?", "answer": "Pros: independent scaling, team autonomy. Cons: complex ops, network overhead. Start monolith, migrate later."},
    {"question": "I feel overwhelmed with work.", "answer": "Write top 3 priorities, tackle hardest first, say no to non-essential tasks. Small steps add up."},
]

# Set up evaluator
judge = CustomLLMJudge(
    provider=LiteLLMProvider(),
    config={"name": "quality", "grading_criteria": "Score 0-1 on helpfulness, conciseness, and tone."},
    model=litellm_model,
)
evaluator = Evaluator(metric=judge)
mapper = BasicDataMapper(key_map={"response": "generated_output", "expected_response": "answer"})

In [ ]:
# Run optimization
initial_prompt = "You are an AI assistant. Answer user questions. Provide information when asked.\n\nUser question: {question}"

teacher = LiteLLMGenerator(model=litellm_model, prompt_template="{prompt}")
optimizer = MetaPromptOptimizer(teacher_generator=teacher)

result = optimizer.optimize(
    evaluator=evaluator,
    data_mapper=mapper,
    dataset=dataset,
    initial_prompts=[initial_prompt],
    task_description="Optimize this prompt to be more helpful, concise, warm, and actionable.",
    num_rounds=3,
    eval_subset_size=len(dataset),
)

print(f"Score: {result.final_score:.4f}")
print(f"\nOptimized prompt:\n{result.best_generator.get_prompt_template()}")

In [ ]:
# Test the optimized prompt
optimized_prompt = result.best_generator.get_prompt_template()

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": optimized_prompt.replace("{question}", "")},
        {"role": "user", "content": "Quick - pros and cons of microservices vs monolith? Meeting in 10 min."}
    ],
)

print("Response with OPTIMIZED prompt:")
print(response.choices[0].message.content)
print("\n--- Compare this to the original response above! ---")